# LP Solvers for Bipartite Graph Matching



In [ ]:
MODE                    = "quick"   # "quick" or "full"
FORCE_REGENERATE_GRAPHS = False
FORCE_RERUN_EXPERIMENTS = False
SOLVERS                 = ["lemon_hk", "highs_lp", "scipy_lp", "gurobi_lp"]

QUICK = dict(sizes=[64, 128, 256], densities=[0.05], seeds=[0, 1, 2], time_limit=60)
FULL  = dict(sizes=[64,128,256,512,1024,2048,4096,8192,16384], densities=[0.01,0.025,0.05,0.075,0.1], seeds=[0,1,2], time_limit=300)

cfg = QUICK if MODE == "quick" else FULL
print(f"Mode: {MODE} | sizes={cfg['sizes']} | densities={cfg['densities']} | time_limit={cfg['time_limit']}s")


In [ ]:
import os, sys, subprocess
from pathlib import Path

ROOT = Path(os.getcwd()).resolve()
assert (ROOT / "python" / "generate_graphs.py").exists(), f"Run from project root. Current dir: {ROOT}"

for d in ["graphs", "results", "figures", "bin", "build"]:
    (ROOT / d).mkdir(exist_ok=True)

GRAPHS_DIR  = ROOT / "graphs"
RESULTS_CSV = ROOT / "results" / "results.csv"
FIGURES_DIR = ROOT / "figures"
PY          = ROOT / "python"

if str(PY) not in sys.path:
    sys.path.insert(0, str(PY))

def run(cmd, **kw):
    r = subprocess.run([str(c) for c in cmd], capture_output=True, text=True, encoding="utf-8", errors="replace", **kw)
    out = ((r.stdout or "") + (r.stderr or "")).strip()
    lines = out.splitlines()
    print("\n".join(lines[-60:] if len(lines) > 60 else lines))
    if r.returncode != 0:
        print(f"[exit {r.returncode}]")
    return r

print(f"Project root: {ROOT}")
print(f"Python: {sys.version.split()[0]}")

## 1. Check Dependencies

In [ ]:
import importlib

PKGS = [
    ("pandas",    True),
    ("matplotlib",True),
    ("numpy",     True),
    ("tqdm",      True),
    ("psutil",    True),
    ("scipy",     False),
    ("highspy",   False),
    ("gurobipy",  False),
]

avail = {}
for pkg, required in PKGS:
    try:
        m = importlib.import_module(pkg)
        avail[pkg] = True
        print(f"  OK       {pkg} {getattr(m, '__version__', '')}")
    except ImportError:
        avail[pkg] = False
        print(f"  {'MISSING (required)' if required else 'missing (optional)'}  {pkg}")

SOLVER_DEPS = {"lemon_hk": True, "scipy_lp": avail.get("scipy"), "highs_lp": avail.get("highspy"), "gurobi_lp": avail.get("gurobipy")}
ACTIVE = [s for s in SOLVERS if SOLVER_DEPS.get(s)]
print(f"\nActive solvers: {ACTIVE}")


## 2. Generate Graphs

In [ ]:
from run_full_benchmark_pipeline import FORCE_REGENERATE_GRAPHS
cmd = [
    sys.executable, PY / "generate_graphs.py",
    "--output-dir", GRAPHS_DIR,
    "--sizes",      *cfg["sizes"],
    "--densities",  *cfg["densities"],
    "--seeds",      *cfg["seeds"],
    "--max-edges",  5_000_000,
]
if FORCE_REGENERATE_GRAPHS:
    cmd.append("--overwrite")

run(cmd)

graphs = sorted(GRAPHS_DIR.glob("*.graph"))
print(f"\nGraph files: {len(graphs)}")
for g in graphs[:5]:
    print(f"  {g.name}")
if len(graphs) > 5:
    print(f"  ... ({len(graphs)-5} more)")

## 3. Build C++ Solvers (optional)

In [ ]:
def find_bin(name):
    for s in ["", ".exe"]:
        p = ROOT / "bin" / f"{name}{s}"
        if p.exists():
            return p
    return None

bins = {n: find_bin(n) for n in ["gurobi_lp", "highs_lp", "lemon_hk"]}
missing = [n for n, p in bins.items() if not p]

if missing and (ROOT / "CMakeLists.txt").exists():
    print("Running CMake build...")
    run(["cmake", "-S", ".", "-B", "build"])
    run(["cmake", "--build", "build", "--config", "Release"])
    bins = {n: find_bin(n) for n in bins}

for n, p in bins.items():
    print(f"  {'FOUND' if p else 'missing'}  {n}")


## 5. Run Experiments

In [ ]:
if not ACTIVE:
    print("No active solvers — skipping")
else:
    cmd = [
        sys.executable, PY / "run_experiments.py",
        "--graphs-dir",     GRAPHS_DIR,
        "--output-csv",     RESULTS_CSV,
        "--time-limit",     cfg["time_limit"],
        "--solvers-python", *ACTIVE,
        "--solvers-cpp",    "none",
        "--sizes",          *cfg["sizes"],
        "--densities",      *cfg["densities"],
        "--seeds",          *cfg["seeds"],
    ]
    if FORCE_RERUN_EXPERIMENTS:
        cmd.append("--overwrite")
    run(cmd)


## 6. Validate Results

In [ ]:
if RESULTS_CSV.exists():
    run([sys.executable, PY / "validate_results.py",
         "--graphs-dir", GRAPHS_DIR, "--results-csv", RESULTS_CSV, "--solvers", *ACTIVE])
else:
    print("No results.csv — run Step 5 first")


## 7. Generate Figures

In [ ]:
if RESULTS_CSV.exists():
    run([sys.executable, PY / "plot_results.py",
         "--input-csv", RESULTS_CSV, "--figures-dir", FIGURES_DIR,
         "--solvers", *ACTIVE, "--format", "pdf", "png"])
else:
    print("No results.csv — run Step 5 first")


## 8. Display Figures

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg

pngs = sorted(FIGURES_DIR.glob("*.png"))
if not pngs:
    print("No PNG figures found")
else:
    for png in pngs:
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.imshow(mpimg.imread(str(png)))
        ax.axis("off")
        ax.set_title(png.stem)
        plt.tight_layout()
        plt.show()


## 9. Results Summary

In [ ]:
import pandas as pd
from IPython.display import display

if not RESULTS_CSV.exists():
    print("No results.csv")
else:
    df = pd.read_csv(RESULTS_CSV)
    if "density" not in df.columns and "d" in df.columns:
        df = df.rename(columns={"d": "density"})

    print(f"Total rows: {len(df)}")

    print("\nStatus counts by solver:")
    display(df.groupby(["solver", "status"]).size().unstack(fill_value=0))

    df_ok = df[df["status"] == "optimal"]
    if not df_ok.empty:
        print("\nMean runtime (s) by solver and n:")
        display(df_ok.groupby(["solver", "n"])["time_seconds"].mean().unstack().round(4))

        print("\nMean peak memory (MB) by solver and n:")
        display(df_ok.groupby(["solver", "n"])["peak_memory_mb"].mean().unstack().round(1))

        print("\nFastest solver per graph:")
        best = df_ok.loc[df_ok.groupby(["n","density","seed"])["time_seconds"].idxmin(), "solver"].value_counts()
        display(best.rename("wins").to_frame())

        print("\nMatching-size agreement (optimal rows):")
        mt = df_ok.groupby(["n","density","seed","solver"])["matching_size"].first().unstack()
        if mt.shape[1] > 1:
            bad = mt[mt.nunique(axis=1) > 1]
            print("All solvers agree." if bad.empty else f"{len(bad)} disagreements:")
            if not bad.empty:
                display(bad)
        else:
            display(mt.head(10))
